# Future Work: Scaling to AfriTeVa-large (Colab)

Self-contained notebook (upload the 4 competition CSVs - `Train.csv`, `Val.csv`,
`Test.csv`, `SampleSubmission.csv` - when prompted; nothing else required). It implements
the report's main future-work direction on hardware larger than the 8 GB laptop GPU used
for the submitted experiments. The Zindi competition has closed, so this is for
demonstration and reproducibility, not for a new leaderboard submission.

### Why a bigger model is the next lever
The public score is `0.37 x ROUGE-1 + 0.37 x ROUGE-L + 0.26 x LLM-judge` (verified exactly
on 8 submissions), and the 3 submission columns are scored independently. The retrieval
baseline caps the ROUGE columns at R1~0.48 / RL~0.40, which limits the total to 0.589 even
with a perfect LLM-judge. Beating that ceiling requires a *generative* model whose own
ROUGE exceeds retrieval. AfriTeVa-V2-base (429M) plateaued below that line on 8 GB, so this
notebook trains the 1B AfriTeVa-V2-large and, per column, uses whichever source scores
higher on Val - so once the model overtakes retrieval on ROUGE, it automatically takes
those columns too.

> GPU note: the 1B model wants bf16 (Colab Pro L4/A100). On a free T4 (fp16 only, where
> T5-family training is unstable) the notebook auto-falls back to AfriTeVa-base in fp32.
> The GPU cell below tells you which path you are on.

## 1. Environment

In [ ]:
!pip -q install -U transformers datasets accelerate sentencepiece rouge_score sacrebleu 2>/dev/null
import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)

In [ ]:
# GPU check + auto precision/model recommendation
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
total_gb = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
print(f'GPU: {name} | {total_gb:.1f} GB | bf16 supported: {bf16_ok}')
if bf16_ok and total_gb >= 20:
    print('=> Recommended: AfriTeVa-large, bf16 (the future-work scaling run).')
elif bf16_ok:
    print('=> AfriTeVa-large in bf16 should fit (L4 ~22GB). Good.')
else:
    print('=> T4/fp16 detected: T5-large fp16 is unstable. Falling back to AfriTeVa-BASE fp32.')
    print('   To run the 1B model, switch to Colab Pro (L4/A100) and set MODEL_NAME=large below.')

## 2. Config
Edit these to switch model / scale. Defaults adapt to your GPU.

In [ ]:
from types import SimpleNamespace
CFG = SimpleNamespace(
    # Large model for the future-work scaling run (needs bf16/Pro). Fallback set automatically for T4.
    model_name = 'castorini/afriteva_v2_large' if bf16_ok else 'castorini/afriteva_v2_base',
    epochs = 4,
    lr = 5e-4,
    max_source_len = 256,
    max_target_len = 384,
    gen_max_new_tokens = 384,
    train_batch_size = 4 if bf16_ok else 8,
    eval_batch_size = 8,
    grad_accum = 4,
    num_beams = 4,
    bf16 = bf16_ok,
    fp16 = False,   # T5-family is unstable in fp16 -> T4 uses fp32 (base model fits 16GB)
    seed = 42,
    max_eval_samples = 500,   # val subset for per-epoch ROUGE (speed)
)
print(CFG)

## 3. Upload the competition CSVs

In [ ]:
import os, pandas as pd
need = ['Train.csv','Val.csv','Test.csv','SampleSubmission.csv']
if not all(os.path.exists(f) for f in need):
    from google.colab import files
    print('Upload:', need); files.upload()
train = pd.read_csv('Train.csv'); val = pd.read_csv('Val.csv')
test = pd.read_csv('Test.csv'); sample = pd.read_csv('SampleSubmission.csv')
print('train', train.shape, '| val', val.shape, '| test', test.shape)

## 4. Preprocessing, prompts & metrics (inline)

In [ ]:
import re, unicodedata
SUBSET_INFO = {'Eng_Uga':('English','Uganda'),'Aka_Gha':('Akan','Ghana'),
  'Eng_Gha':('English','Ghana'),'Eng_Eth':('English','Ethiopia'),'Lug_Uga':('Luganda','Uganda'),
  'Eng_Ken':('English','Kenya'),'Swa_Ken':('Swahili','Kenya'),'Amh_Eth':('Amharic','Ethiopia')}
_WS = re.compile(r'[ \t\u00a0]+')
def clean_text(t):
    if t is None: return ''
    t = unicodedata.normalize('NFC', str(t)).replace('\r\n','\n').replace('\r','\n')
    return _WS.sub(' ', t).strip()
def build_source(q, s):
    return f'<{s}> answer health question: ' + clean_text(q)

# Multilingual ROUGE (no stemmer, unicode word tokenizer)
from rouge_score import rouge_scorer
from rouge_score.tokenizers import Tokenizer
_TOK = re.compile(r'\w+', re.UNICODE)
class _UT(Tokenizer):
    def tokenize(self, text): return _TOK.findall(text.lower())
_SC = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=False, tokenizer=_UT())
def compute_rouge(preds, refs):
    r1=rl=0.0; n=max(len(preds),1)
    for p,r in zip(preds,refs):
        s=_SC.score(r or '', p or ''); r1+=s['rouge1'].fmeasure; rl+=s['rougeL'].fmeasure
    r1/=n; rl/=n; return {'rouge1_f1':r1,'rougeL_f1':rl,'mean_f1':(r1+rl)/2}

## 5. Retrieval baseline for the ROUGE columns
Char n-gram TF-IDF nearest-neighbour per subset. We keep its Val score so we can
later decide, per column, whether the trained model beats it.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import numpy as np
for d in (train,val,test): d['q']=d['input'].map(clean_text)
def retrieve(src_df, q_df):
    pred=pd.Series(index=q_df.index,dtype='object')
    for s,g in q_df.groupby('subset'):
        t=src_df[src_df.subset==s]
        if len(t)==0: t=src_df
        v=TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1)
        tm=v.fit_transform(t['q']); qm=v.transform(g['q'])
        pred.loc[g.index]=t['output'].to_numpy()[linear_kernel(qm,tm).argmax(1)]
    return pred
retr_test = retrieve(train, test)
retr_val  = retrieve(train, val)
rv = compute_rouge(retr_val.tolist(), val['output'].tolist())
print('Retrieval Val: R1=%.4f RL=%.4f'%(rv['rouge1_f1'],rv['rougeL_f1']))

## 6. Build datasets & tokenize

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(CFG.model_name)
def to_ds(df, with_t=True):
    d={'source':[build_source(q,s) for q,s in zip(df['input'],df['subset'])], 'subset':list(df['subset'])}
    if with_t: d['target']=[clean_text(a) for a in df['output']]
    return Dataset.from_dict(d)
raw = DatasetDict(train=to_ds(train), val=to_ds(val), test=to_ds(test, False))
def tk(b):
    m=tok(b['source'], max_length=CFG.max_source_len, truncation=True)
    if 'target' in b: m['labels']=tok(text_target=b['target'], max_length=CFG.max_target_len, truncation=True)['input_ids']
    return m
ds = DatasetDict({k: raw[k].map(tk, batched=True, remove_columns=[c for c in raw[k].column_names if c!='subset']) for k in raw})
eval_ds = ds['val'].select(range(min(CFG.max_eval_samples, len(ds['val']))))

## 7. Train
Per-epoch Val ROUGE is printed so you can watch the model approach (and ideally
pass) the retrieval ROUGE line. **Watch `rouge1_f1` vs the retrieval R1 above.**

In [ ]:
import numpy as np
from transformers import (AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments)
torch.manual_seed(CFG.seed)
model = AutoModelForSeq2SeqLM.from_pretrained(CFG.model_name)
model.gradient_checkpointing_enable(); model.config.use_cache=False
collator = DataCollatorForSeq2Seq(tok, model=model, padding='longest', label_pad_token_id=-100)
def metrics(ep):
    preds,labels=ep
    if isinstance(preds,tuple): preds=preds[0]
    preds=np.where(preds!=-100,preds,tok.pad_token_id); labels=np.where(labels!=-100,labels,tok.pad_token_id)
    dp=[t.replace('<extra_id_0>','').strip() for t in tok.batch_decode(preds,skip_special_tokens=True)]
    dr=tok.batch_decode(labels,skip_special_tokens=True)
    return compute_rouge(dp,dr)
args = Seq2SeqTrainingArguments(output_dir='out', num_train_epochs=CFG.epochs, learning_rate=CFG.lr,
    per_device_train_batch_size=CFG.train_batch_size, per_device_eval_batch_size=CFG.eval_batch_size,
    gradient_accumulation_steps=CFG.grad_accum, warmup_ratio=0.05, bf16=CFG.bf16, fp16=CFG.fp16,
    optim='adafactor', predict_with_generate=True, generation_max_length=CFG.gen_max_new_tokens,
    generation_num_beams=CFG.num_beams, eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='mean_f1', greater_is_better=True,
    logging_steps=50, report_to='none', seed=CFG.seed)
import inspect
tkw = 'processing_class' if 'processing_class' in inspect.signature(Seq2SeqTrainer.__init__).parameters else 'tokenizer'
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=eval_ds,
    data_collator=collator, compute_metrics=metrics, **{tkw: tok})
trainer.train()
print('best val:', trainer.evaluate())

## 8. Evaluate the model on Val (decide column ownership)

In [ ]:
def gen(srcs, mx=CFG.gen_max_new_tokens, bs=16):
    model.eval(); out=[]
    for i in range(0,len(srcs),bs):
        enc=tok(srcs[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=CFG.max_source_len).to(model.device)
        with torch.no_grad():
            g=model.generate(**enc, max_new_tokens=mx, num_beams=CFG.num_beams, no_repeat_ngram_size=3)
        out+= [t.replace('<extra_id_0>','').strip() for t in tok.batch_decode(g, skip_special_tokens=True)]
    return out
val_src=[build_source(q,s) for q,s in zip(val['input'],val['subset'])]
model_val = gen(val_src)
mv = compute_rouge(model_val, val['output'].tolist())
print('MODEL Val:     R1=%.4f RL=%.4f mean=%.4f'%(mv['rouge1_f1'],mv['rougeL_f1'],mv['mean_f1']))
print('RETRIEVAL Val: R1=%.4f RL=%.4f'%(rv['rouge1_f1'],rv['rougeL_f1']))
use_model_r1 = mv['rouge1_f1'] > rv['rouge1_f1']
use_model_rl = mv['rougeL_f1'] > rv['rougeL_f1']
print('=> ROUGE-1 column uses:', 'MODEL' if use_model_r1 else 'retrieval')
print('=> ROUGE-L column uses:', 'MODEL' if use_model_rl else 'retrieval')
print('=> LLM column always uses MODEL')

## 9. Generate test predictions & assemble the submission
Each column gets whichever source scored higher on Val - the per-column optimization.

In [ ]:
test_src=[build_source(q,s) for q,s in zip(test['input'],test['subset'])]
model_test = gen(test_src)
model_test = [t if t.strip() else 'No information available.' for t in model_test]
sub = pd.DataFrame({'ID': test['ID']})
sub['TargetR1F1'] = list(model_test) if use_model_r1 else retr_test.values
sub['TargetRLF1'] = list(model_test) if use_model_rl else retr_test.values
sub['TargetLLM']  = list(model_test)
sub = sub[['ID','TargetRLF1','TargetR1F1','TargetLLM']]
assert (sub['ID'].values==sample['ID'].values).all() and sub.isna().sum().sum()==0
fname = 'submission_pushto06.csv'; sub.to_csv(fname, index=False, encoding='utf-8')
# Expected public score from the verified formula (val proxy for LLM is unknown; shows ROUGE part)
r1 = mv['rouge1_f1'] if use_model_r1 else rv['rouge1_f1']
rl = mv['rougeL_f1'] if use_model_rl else rv['rougeL_f1']
print('Wrote', fname)
print('Val ROUGE part of score: 0.37*%.3f + 0.37*%.3f = %.3f  (+0.26*LLM on the leaderboard)'%(r1,rl,0.37*r1+0.37*rl))

## 10. Download & submit

In [ ]:
from google.colab import files
files.download('submission_pushto06.csv')
# Upload to Zindi (Submissions tab). Comment suggestion:
print('Comment: %s, %d epochs, per-column (model owns ROUGE cols: R1=%s RL=%s)'%(
    CFG.model_name.split('/')[-1], CFG.epochs, use_model_r1, use_model_rl))

## Notes & honest expectations
- If the model's Val R1 stays below retrieval (~0.48), the ROUGE columns keep using
  retrieval and the total stays near ~0.47 (the 0.589 ceiling is not broken). Levers: more
  epochs, the large model (needs Pro), longer targets, or better data handling.
- If the model's Val R1 passes the retrieval line (~0.48 and up), the model takes the
  ROUGE columns too and the total climbs past 0.589. That is the milestone to watch in cell 8.
- Save the model to Drive to resume across Colab sessions:
  `from google.colab import drive; drive.mount('/content/drive'); trainer.save_model('/content/drive/MyDrive/healthqa_model')`
- Reproducibility: seed is fixed; the config block fully determines the run.